# Warehouse Layer EDA — `warehouse` schema
End-to-end marts analysis on `dim_customers`, `dim_products` and
`fact_sales`: grain checks, revenue cuts, top movers, calendar effects,
price structure and SCD behavior. Mirrors `tests/data_quality/warehouse/`.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psycopg
import seaborn as sns
from dotenv import load_dotenv

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy connectable")

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(REPO_ROOT / ".env")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)
plt.rcParams["figure.figsize"] = (10, 5)

WH = "warehouse"
STG = "staging"


def pg_conn():
    return psycopg.connect(
        host=os.getenv("POSTGRES_HOST", "localhost"),
        port=os.getenv("POSTGRES_PORT", "5432"),
        dbname=os.getenv("POSTGRES_DB", "datawarehouse"),
        user=os.getenv("POSTGRES_USER", "postgres"),
        password=os.getenv("POSTGRES_PASSWORD", ""),
        autocommit=True,
    )


def read_table(name, schema):
    with pg_conn() as conn:
        return pd.read_sql_query(f'SELECT * FROM "{schema}"."{name}"', conn)


cust = read_table("dim_customers", WH)
prd = read_table("dim_products", WH)
fact = read_table("fact_sales", WH)
print("warehouse loaded:", {"dim_customers": cust.shape, "dim_products": prd.shape, "fact_sales": fact.shape})

enriched = fact.merge(cust[["customer_sk", "country"]], left_on="customer_key", right_on="customer_sk", how="left")
enriched = enriched.merge(prd[["product_sk", "category", "subcategory", "product_name"]],
                          left_on="product_key", right_on="product_sk", how="left")
enriched["order_date"] = pd.to_datetime(enriched["order_date"], errors="coerce")
for c in ["sales_amount", "quantity", "price"]:
    enriched[c] = pd.to_numeric(enriched[c], errors="coerce")
print("enriched:", enriched.shape, "| null FKs:",
      int(enriched["customer_sk"].isna().sum() + enriched["product_sk"].isna().sum()))


## 1. Inventory and grain — dims plus one row per sales line.

In [ ]:
inv = pd.DataFrame([
    {"table": "dim_customers", "rows": len(cust)},
    {"table": "dim_products", "rows": len(prd)},
    {"table": "fact_sales", "rows": len(fact)},
    {"table": "staging.sales_details", "rows": read_table("sales_details", STG).shape[0]},
])
inv
print("total revenue:", f"{enriched['sales_amount'].sum():,.0f}")
print("orders:", enriched["order_number"].nunique(), "| customers buying:",
      enriched["customer_key"].nunique(), "| products sold:", enriched["product_key"].nunique())


## 2. Revenue cuts — month trend, category and country bars.

In [ ]:
enriched["order_month"] = enriched["order_date"].dt.to_period("M").astype(str)
enriched.groupby("order_month")["sales_amount"].sum().plot(figsize=(14, 4), title="Revenue per month")
plt.tight_layout()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
enriched.groupby("category")["sales_amount"].sum().sort_values().plot.barh(
    ax=axes[0], title="Revenue by category")
enriched.groupby("country")["sales_amount"].sum().sort_values().tail(10).plot.barh(
    ax=axes[1], title="Revenue by country (top 10)")
plt.tight_layout()


## 3. Top movers — products and customers.

In [ ]:
top_prd = enriched.groupby(["product_name", "category"])["sales_amount"].sum().sort_values(ascending=False).head(10)
print(top_prd.to_string())
top_prd.sort_values().plot.barh(figsize=(9, 5), title="Top 10 products by revenue")
plt.tight_layout()

top_cust = enriched.groupby("customer_key")["sales_amount"].sum().sort_values(ascending=False).head(10)
print(top_cust.to_string())


## 4. Calendar — weekday mix and ship lag.

In [ ]:
enriched["weekday"] = enriched["order_date"].dt.day_name()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
enriched["weekday"].value_counts().reindex(order).plot.bar(ax=axes[0], title="Orders by weekday")
enriched.groupby("weekday")["sales_amount"].sum().reindex(order).plot.bar(
    ax=axes[1], title="Revenue by weekday", color="green")
plt.tight_layout()

enriched["ship_lag"] = (pd.to_datetime(enriched["shipping_date"], errors="coerce") - enriched["order_date"]).dt.days
print("avg ship lag days:", round(enriched["ship_lag"].mean(), 2))


## 5. Price structure — boxplots by category plus cost margin.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.boxplot(data=enriched, x="price", y="category", ax=axes[0])
axes[0].set_title("Price boxplot by category")
enriched["cost"] = enriched["product_sk"].map(prd.set_index("product_sk")["cost"])
enriched["cost"] = pd.to_numeric(enriched["cost"], errors="coerce")
enriched["margin"] = enriched["sales_amount"] - enriched["quantity"] * enriched["cost"]
print("avg margin per line:", round(enriched["margin"].mean(), 2))
print(enriched.groupby("category")["margin"].mean().round(2).sort_values(ascending=False).to_string())
plt.tight_layout()
print(enriched.groupby("category")["price"].describe().round(2).to_string())


## 6. SCD and coverage — dim freshness, orphans, grain proof.

In [ ]:
print("dim_customers updated_at distinct:", cust["updated_at"].nunique())
print("dim_products updated_at distinct:", prd["updated_at"].nunique())
print("fact rows == staging rows:", len(fact) == read_table("sales_details", STG).shape[0])
print("products with category:", prd["category"].notna().sum(), "/", len(prd))
print("customers with country:", cust["country"].notna().sum(), "/", len(cust))
print("fact customers in dim:", enriched["customer_key"].nunique(), "of", cust.shape[0])
